# Geocode Addresses (ArcGIS)

This notebook demonstrates how to use the `geocoding-kit` module with `arcgis` / `arcgis-mapping` to geocode a list of addresses (from `addresses.csv`) and display the results.

**Prerequisites:**
- Install dependencies: `pip install arcgis arcgis-mapping pandas`
- Set `ARCGIS_API_KEY` in your environment or `.env` file.

Run the cells sequentially to see geocoding results.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from arcgis.gis import GIS
import arcgis.mapping as mapping

# If you have not installed the local `geocoding_kit` package, add the repository root to sys.path.
# This allows running the notebook directly from the `examples/` folder.
repo_root = Path().resolve().parent
sys.path.insert(0, str(repo_root))

from geocoding_kit import ArcGISConfig, PlatformGeocoder
from geocoding_kit.models import AddressInput

# Ensure the API key is set (either in the environment or via a .env file)
api_key = os.getenv('ARCGIS_API_KEY')
if not api_key:
    raise ValueError('Missing ARCGIS_API_KEY environment variable. Set it before running the notebook.')

# Create a GIS instance (optional, used to show how to use arcgis and arcgis-mapping)
gis = GIS(api_key=api_key)
map_object = gis.map('World')
map_object.title = 'Geocoded Addresses (ArcGIS)'
map_object

## Load Address Data

Load the sample addresses from `addresses.csv`. The file must include at least the columns: `id`, `address`, `postal`, `city`, and `country`.

In [ ]:
address_csv = 'addresses.csv'

df = pd.read_csv(address_csv)
df.head()

## Prepare Inputs for Geocoding

Convert the CSV rows into `AddressInput` instances used by `geocoding-kit`.

In [ ]:
addresses = [
    AddressInput(
        id=int(row['id']),
        address=str(row['address']),
        postal=str(row.get('postal', '')),
        city=str(row.get('city', '')),
        country=str(row.get('country', '')),
    )
    for _, row in df.iterrows()
]

len(addresses), 'addresses loaded'

## Run Geocoding

Use the `PlatformGeocoder` from `geocoding-kit` to geocode the address list.

In [ ]:
config = ArcGISConfig.from_env()  # reads ARCGIS_API_KEY and optional ARCGIS_GEOCODE_URL
geocoder = PlatformGeocoder(config)

results = geocoder.geocode(addresses)

results_df = pd.DataFrame([
    {
        'id': r.id,
        'input_address': r.input_address,
        'matched_address': r.matched_address,
        'latitude': r.latitude,
        'longitude': r.longitude,
        'score': r.score,
        'match_status': r.match_status,
    }
    for r in results
])

results_df.head()

## Display Results on a Map (Optional)

If the geocoding succeeded, you can add the points to the map created above.

In [ ]:
# Add points to the map if coordinates are available
points = results_df.dropna(subset=['latitude', 'longitude'])

if not points.empty:
    features = [
        {
            'geometry': {'x': float(row['longitude']), 'y': float(row['latitude'])},
            'attributes': {
                'matched_address': row['matched_address'],
                'score': row['score'],
            },
        }
        for _, row in points.iterrows()
    ]

    from arcgis.features import FeatureSet
    fs = FeatureSet(features)
    map_object.add_layer(fs, {'renderer': {'type': 'simple', 'symbol': {'type': 'esriSMS','style': 'esriSMSCircle','color': [0, 0, 255, 128],'size': 10}}})
    map_object
else:
    print('No geocoded points available to display on the map.')